# Experiment 1: Cellpose Hybrid Detection Pipeline

This notebook replaces the classical threshold detector with **Cellpose** — a deep-learning
instance segmentation model pretrained on diverse cell microscopy data.

**Three detectors compared**:
1. `threshold` — Gaussian smooth + percentile threshold + connected components *(baseline)*
2. `cellpose` — Cellpose `nuclei` model in 2D-slice mode *(pure DL)*
3. `hybrid` — Run both, merge centroids with NMS *(best of both worlds)*

All evaluated against ground-truth `.geff` files using the competition metric.

## Install Cellpose
Run this cell once. Cellpose depends on PyTorch (CPU-only is fine).

In [ ]:
# Uncomment and run if cellpose is not already installed
# !pip install cellpose

## Imports

In [ ]:
import json
import os
import time
import traceback

import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# Cellpose
try:
    from cellpose import models as cp_models
    CELLPOSE_AVAILABLE = True
    print("Cellpose loaded successfully.")
except ImportError:
    CELLPOSE_AVAILABLE = False
    print("WARNING: Cellpose not installed. Only classical detector will be available.")
    print("Install with: pip install cellpose")

# Chunk decompressor
try:
    from numcodecs import Blosc as _NumcodecsBlosc
    _USE_NUMCODECS = True
except ImportError:
    _USE_NUMCODECS = False
    import blosc2

## Configuration

In [ ]:
# Local paths — change these to match your setup
DATA_DIR = "f:/4D cell tracking/biohub-cell-tracking-during-development"
SPLIT = "train"  # evaluate against train set ground truth

# How many samples to evaluate (set to None for all)
MAX_SAMPLES = 3

# Physical voxel scale in micrometers (from competition spec)
VOXEL_SCALE = np.array([1.625, 0.40625, 0.40625])  # z, y, x

# Anisotropy ratio for Cellpose: z_spacing / xy_spacing
ANISOTROPY = VOXEL_SCALE[0] / VOXEL_SCALE[1]  # = 4.0

# Classical detector parameters
DETECTION_SIGMA_UM = 2.0
DETECTION_THRESHOLD_PCT = 85.0
DETECTION_MIN_SIZE = 10

# Tracker parameters
MAX_LINK_DISTANCE_UM = 7.0

# Hybrid merge: NMS distance in micrometers
NMS_DISTANCE_UM = 3.0

print(f"Data dir: {DATA_DIR}")
print(f"Anisotropy (z/xy): {ANISOTROPY:.1f}")

## Data I/O

In [ ]:
def get_array_meta(zarr_path: str) -> dict:
    """Read the array-level zarr.json at <zarr>/0/zarr.json."""
    with open(os.path.join(zarr_path, "0", "zarr.json"), "r") as f:
        return json.load(f)


def load_volume_frame(zarr_path: str, t: int) -> np.ndarray:
    """Load a single timepoint (Z, Y, X) uint16 from a Zarr v3 store."""
    meta = get_array_meta(zarr_path)
    shape = tuple(meta["shape"])
    chunk_shape = tuple(meta["chunk_grid"]["configuration"]["chunk_shape"])
    dtype = np.dtype(meta["data_type"]).newbyteorder("<")
    chunk_path = os.path.join(zarr_path, "0", "c", str(t), "0", "0", "0")
    with open(chunk_path, "rb") as f:
        raw = f.read()
    if _USE_NUMCODECS:
        codec = _NumcodecsBlosc(cname="zstd", clevel=1, shuffle=_NumcodecsBlosc.BITSHUFFLE)
        buf = codec.decode(raw)
    else:
        buf = blosc2.decompress(raw)
    return np.frombuffer(buf, dtype=dtype).reshape(chunk_shape)[0]


def voxel_to_physical(coords_zyx: np.ndarray) -> np.ndarray:
    """Convert voxel (z,y,x) coordinates to physical micrometers."""
    return coords_zyx * VOXEL_SCALE


def load_geff(geff_path: str) -> dict:
    """Load a .geff ground-truth graph."""
    from numcodecs import Zstd
    meta_file = os.path.join(geff_path, "zarr.json")
    with open(meta_file, "r") as f:
        meta = json.load(f)

    def read_arr(path, dtype):
        zstd = Zstd(level=0)
        cf = os.path.join(path, "c", "0")
        if not os.path.isfile(cf):
            cf = os.path.join(path, "c", "0", "0")
        with open(cf, "rb") as f:
            raw = f.read()
        return np.frombuffer(zstd.decode(raw), dtype=dtype)

    node_ids = read_arr(os.path.join(geff_path, "nodes", "ids"), "<u8")
    props = {}
    for prop in ["t", "z", "y", "x"]:
        props[prop] = read_arr(
            os.path.join(geff_path, "nodes", "props", prop, "values"), "<i8"
        )
    nodes_df = pd.DataFrame({"node_id": node_ids, **props})

    edges_path = os.path.join(geff_path, "edges", "ids")
    with open(os.path.join(edges_path, "zarr.json"), "r") as f:
        n_edges = json.load(f)["shape"][0]
    edges = read_arr(edges_path, "<u8").reshape(n_edges, 2)

    return {"node_ids": node_ids, "nodes_df": nodes_df, "edges": edges, "metadata": meta}


def list_samples(data_dir: str, split: str) -> list:
    """List sample names in a split directory."""
    split_dir = os.path.join(data_dir, split)
    return sorted(set(e.replace(".zarr", "") for e in os.listdir(split_dir) if e.endswith(".zarr")))

## Detector 1: Classical Threshold (Baseline)

In [ ]:
def detect_cells_threshold(
    volume_3d: np.ndarray,
    voxel_scale: np.ndarray = None,
    sigma_um: float = DETECTION_SIGMA_UM,
    threshold_percentile: float = DETECTION_THRESHOLD_PCT,
    min_size_voxels: int = DETECTION_MIN_SIZE,
) -> np.ndarray:
    """Gaussian smooth + threshold + connected components.
    Returns (N, 3) centroids in voxel coords (z, y, x).
    """
    if voxel_scale is None:
        voxel_scale = VOXEL_SCALE
    sigma_voxels = sigma_um / voxel_scale
    smoothed = ndimage.gaussian_filter(volume_3d.astype(np.float32), sigma=sigma_voxels)
    thresh = np.percentile(smoothed, threshold_percentile)
    labeled, n_labels = ndimage.label(smoothed > thresh)
    centroids = []
    for i in range(1, n_labels + 1):
        if (labeled == i).sum() < min_size_voxels:
            continue
        centroids.append(ndimage.center_of_mass(smoothed, labeled, i))
    if len(centroids) == 0:
        return np.empty((0, 3), dtype=np.float64)
    return np.array(centroids, dtype=np.float64)

## Detector 2: Cellpose (Deep Learning)

Uses the pretrained `nuclei` model in **2D per-slice** mode for speed on CPU.
Each Z-slice is segmented independently, then instance masks are stitched
across Z using 3D connected components on the label overlaps.

This is ~10-50× faster than full `do_3D=True` mode while still leveraging
learned features for each XY plane.

In [ ]:
def detect_cells_cellpose(
    volume_3d: np.ndarray,
    voxel_scale: np.ndarray = None,
    model_type: str = "nuclei",
    diameter: float = None,
    flow_threshold: float = 0.4,
    cellprob_threshold: float = 0.0,
    min_size: int = 15,
) -> np.ndarray:
    """Detect cells using Cellpose in 2D per-slice mode, then stitch in 3D.

    Parameters
    ----------
    volume_3d : (Z, Y, X) uint16 or float
    model_type : str
        Cellpose model: 'nuclei', 'cyto', 'cyto3', etc.
    diameter : float or None
        Expected cell diameter in pixels (XY). None = auto-estimate.
    flow_threshold : float
        Cellpose flow error threshold. Higher = more permissive.
    cellprob_threshold : float
        Minimum cell probability to keep.
    min_size : int
        Minimum mask size in pixels (per 2D slice).

    Returns
    -------
    np.ndarray, shape (N, 3) — centroids in voxel coords (z, y, x)
    """
    if not CELLPOSE_AVAILABLE:
        raise RuntimeError("Cellpose is not installed.")

    if voxel_scale is None:
        voxel_scale = VOXEL_SCALE

    model = cp_models.Cellpose(gpu=False, model_type=model_type)

    Z, Y, X = volume_3d.shape

    # Normalize volume to [0, 255] uint8 for Cellpose
    vol = volume_3d.astype(np.float32)
    p_low, p_high = np.percentile(vol, [1, 99.5])
    if p_high <= p_low:
        p_high = p_low + 1
    vol = np.clip((vol - p_low) / (p_high - p_low) * 255, 0, 255).astype(np.uint8)

    # --- Step 1: Run Cellpose on each Z-slice independently ---
    slice_labels = np.zeros((Z, Y, X), dtype=np.int32)
    global_label_offset = 0

    for z in range(Z):
        img_2d = vol[z]  # (Y, X)

        masks, _, _, _ = model.eval(
            img_2d,
            diameter=diameter,
            channels=[0, 0],  # grayscale
            flow_threshold=flow_threshold,
            cellprob_threshold=cellprob_threshold,
            min_size=min_size,
        )

        if masks.max() > 0:
            # Offset labels so they're unique across slices
            masks_offset = np.where(masks > 0, masks + global_label_offset, 0)
            slice_labels[z] = masks_offset
            global_label_offset += masks.max()

    if slice_labels.max() == 0:
        return np.empty((0, 3), dtype=np.float64)

    # --- Step 2: Stitch across Z using 3D connected components ---
    # Relabel so that touching labels across Z-slices merge
    binary = slice_labels > 0
    stitched, n_stitched = ndimage.label(binary)

    # --- Step 3: Extract centroids ---
    centroids = []
    for i in range(1, n_stitched + 1):
        mask_size = (stitched == i).sum()
        if mask_size < min_size:
            continue
        com = ndimage.center_of_mass(vol.astype(np.float32), stitched, i)
        centroids.append(com)

    if len(centroids) == 0:
        return np.empty((0, 3), dtype=np.float64)

    return np.array(centroids, dtype=np.float64)

## Detector 3: Hybrid (Classical + Cellpose Merge)

Strategy: run both detectors, merge their centroids, and apply
Non-Maximum Suppression (NMS) in physical space to remove duplicates.

In [ ]:
def nms_centroids(
    centroids: np.ndarray,
    voxel_scale: np.ndarray,
    min_distance_um: float = NMS_DISTANCE_UM,
) -> np.ndarray:
    """Non-Maximum Suppression: remove duplicate centroids that are
    within min_distance_um of each other in physical space.
    Keeps the first occurrence (priority order matters).
    """
    if len(centroids) <= 1:
        return centroids

    phys = centroids * voxel_scale
    keep = np.ones(len(centroids), dtype=bool)

    for i in range(len(centroids)):
        if not keep[i]:
            continue
        for j in range(i + 1, len(centroids)):
            if not keep[j]:
                continue
            dist = np.linalg.norm(phys[i] - phys[j])
            if dist < min_distance_um:
                keep[j] = False  # suppress the later one

    return centroids[keep]


def detect_cells_hybrid(
    volume_3d: np.ndarray,
    voxel_scale: np.ndarray = None,
    nms_distance_um: float = NMS_DISTANCE_UM,
) -> np.ndarray:
    """Hybrid detector: classical threshold + Cellpose, merged with NMS.

    Classical detections are placed first (higher priority in NMS),
    Cellpose detections fill in cells that classical missed.
    """
    if voxel_scale is None:
        voxel_scale = VOXEL_SCALE

    # Classical (fast, reliable for bright cells)
    c_classical = detect_cells_threshold(volume_3d, voxel_scale)

    # Cellpose (slower, catches dim/touching cells)
    if CELLPOSE_AVAILABLE:
        c_cellpose = detect_cells_cellpose(volume_3d, voxel_scale)
    else:
        c_cellpose = np.empty((0, 3), dtype=np.float64)

    # Merge: classical first (priority), then cellpose additions
    if len(c_classical) > 0 and len(c_cellpose) > 0:
        merged = np.vstack([c_classical, c_cellpose])
    elif len(c_cellpose) > 0:
        merged = c_cellpose
    else:
        merged = c_classical

    # NMS to remove duplicates
    return nms_centroids(merged, voxel_scale, nms_distance_um)

## Tracker

Hungarian frame-to-frame linking + division detection. Identical to the
modular `tracker.py` — the tracker is agnostic to which detector produced
the centroids.

In [ ]:
class TrackGraph:
    """Simple tracking graph: nodes + directed edges."""
    def __init__(self):
        self.nodes = []
        self.edges = []
        self._next_id = 1

    def add_node(self, t, z, y, x):
        nid = self._next_id; self._next_id += 1
        self.nodes.append({"id": nid, "t": t, "z": int(round(z)), "y": int(round(y)), "x": int(round(x))})
        return nid

    def add_edge(self, src, tgt):
        self.edges.append((src, tgt))

    def get_nodes_at_time(self, t):
        return [n for n in self.nodes if n["t"] == t]

    def get_outgoing_edges(self, nid):
        return [tgt for src, tgt in self.edges if src == nid]

    def get_incoming_edges(self, nid):
        return [src for src, tgt in self.edges if tgt == nid]


def link_frames(c0, c1, voxel_scale=None, max_dist=MAX_LINK_DISTANCE_UM):
    """Hungarian matching between two frames. Returns list of (i, j) pairs."""
    if voxel_scale is None: voxel_scale = VOXEL_SCALE
    if len(c0) == 0 or len(c1) == 0: return []
    cost = cdist(c0 * voxel_scale, c1 * voxel_scale)
    cost_m = np.where(cost <= max_dist, cost, 1e6)
    ri, ci = linear_sum_assignment(cost_m)
    return [(int(i), int(j)) for i, j in zip(ri, ci) if cost[i, j] <= max_dist]


def build_tracks(all_centroids, voxel_scale=None, max_dist=MAX_LINK_DISTANCE_UM):
    if voxel_scale is None: voxel_scale = VOXEL_SCALE
    graph = TrackGraph()
    timepoints = sorted(all_centroids.keys())
    if not timepoints: return graph
    frame_ids = {}
    for t in timepoints:
        ids = [graph.add_node(t, *c) for c in all_centroids[t]]
        frame_ids[t] = ids
    for i in range(len(timepoints) - 1):
        t0, t1 = timepoints[i], timepoints[i+1]
        if t1 != t0 + 1: continue
        for i0, i1 in link_frames(all_centroids[t0], all_centroids[t1], voxel_scale, max_dist):
            graph.add_edge(frame_ids[t0][i0], frame_ids[t1][i1])
    return graph


def detect_divisions(graph, all_centroids, voxel_scale=None, max_dist=MAX_LINK_DISTANCE_UM):
    if voxel_scale is None: voxel_scale = VOXEL_SCALE
    timepoints = sorted(all_centroids.keys())
    for i in range(1, len(timepoints)):
        t, tp = timepoints[i], timepoints[i-1]
        if t != tp + 1: continue
        nodes_t = graph.get_nodes_at_time(t)
        unmatched = [n for n in nodes_t if not graph.get_incoming_edges(n["id"])]
        if not unmatched: continue
        nodes_prev = graph.get_nodes_at_time(tp)
        if not nodes_prev: continue
        p_coords = np.array([[n["z"], n["y"], n["x"]] for n in nodes_prev], dtype=float)
        p_ids = [n["id"] for n in nodes_prev]
        p_phys = p_coords * voxel_scale
        for node in unmatched:
            c_phys = np.array([[node["z"], node["y"], node["x"]]], dtype=float) * voxel_scale
            dists = cdist(c_phys, p_phys)[0]
            idx = np.argmin(dists)
            if dists[idx] <= max_dist and len(graph.get_outgoing_edges(p_ids[idx])) == 1:
                graph.add_edge(p_ids[idx], node["id"])
    return graph

## Evaluation Metrics

Competition metric: Edge Jaccard + Division Jaccard, computed by matching
predicted nodes to GT nodes per timepoint via bipartite assignment.

In [ ]:
def _match_nodes_at_time(pred_nodes, gt_nodes, voxel_scale, max_dist_um=7.0):
    """Match pred -> GT nodes at one timepoint. Returns dict pred_id -> gt_id."""
    if not pred_nodes or not gt_nodes: return {}
    pc = np.array([[n["z"], n["y"], n["x"]] for n in pred_nodes], dtype=float)
    gc = np.array([[n["z"], n["y"], n["x"]] for n in gt_nodes], dtype=float)
    cost = cdist(pc * voxel_scale, gc * voxel_scale)
    cost_m = np.where(cost <= max_dist_um, cost, 1e6)
    ri, ci = linear_sum_assignment(cost_m)
    return {pred_nodes[i]["id"]: gt_nodes[j]["node_id"]
            for i, j in zip(ri, ci) if cost[i, j] <= max_dist_um}


def evaluate_sample(pred_graph, gt_geff, voxel_scale=None, max_dist_um=7.0):
    """Evaluate one sample: returns dict with edge/division Jaccard scores."""
    if voxel_scale is None: voxel_scale = VOXEL_SCALE
    gt_df = gt_geff["nodes_df"]
    gt_edges_arr = gt_geff["edges"]
    est_nodes = gt_geff["metadata"]["attributes"]["geff"]["extra"]["estimated_number_of_nodes"]

    gt_by_t = {}
    for _, row in gt_df.iterrows():
        t = int(row["t"])
        gt_by_t.setdefault(t, []).append(
            {"node_id": int(row["node_id"]), "z": int(row["z"]), "y": int(row["y"]), "x": int(row["x"])})

    pred_by_t = {}
    for n in pred_graph.nodes:
        pred_by_t.setdefault(n["t"], []).append(n)

    pred_to_gt = {}
    for t in sorted(set(list(gt_by_t) + list(pred_by_t))):
        pred_to_gt.update(_match_nodes_at_time(
            pred_by_t.get(t, []), gt_by_t.get(t, []), voxel_scale, max_dist_um))

    # Edge Jaccard
    gt_edge_set = {(int(e[0]), int(e[1])) for e in gt_edges_arr}
    pred_mapped = set()
    for s, tg in pred_graph.edges:
        gs, gtg = pred_to_gt.get(s), pred_to_gt.get(tg)
        if gs is not None and gtg is not None:
            pred_mapped.add((gs, gtg))

    etp = len(pred_mapped & gt_edge_set)
    efp = len(pred_mapped - gt_edge_set)
    efn = len(gt_edge_set - pred_mapped)
    overcount = max(0, len(pred_graph.nodes) - est_nodes)
    ed = etp + efp + efn + overcount
    ej = etp / ed if ed > 0 else 0.0

    # Division Jaccard
    gt_out = {}
    for s, tg in gt_edge_set:
        gt_out.setdefault(s, []).append(tg)
    gt_divs = {s: ts for s, ts in gt_out.items() if len(ts) >= 2}

    pred_out = {}
    for s, tg in pred_graph.edges:
        pred_out.setdefault(s, []).append(tg)
    pred_divs = {s: ts for s, ts in pred_out.items() if len(ts) >= 2}

    dtp = 0; matched_gt = set()
    for pp, pds in pred_divs.items():
        gp = pred_to_gt.get(pp)
        if gp is None or gp not in gt_divs: continue
        mapped_d = {pred_to_gt.get(d) for d in pds} - {None}
        if len(mapped_d & set(gt_divs[gp])) >= 2:
            dtp += 1; matched_gt.add(gp)

    dfn = len(gt_divs) - len(matched_gt)
    dfp = sum(1 for p in pred_divs if pred_to_gt.get(p) not in matched_gt)
    dd = dtp + dfp + dfn
    dj = dtp / dd if dd > 0 else 0.0

    return {"edge_tp": etp, "edge_fp": efp, "edge_fn": efn,
            "edge_jaccard": ej, "node_overcount": overcount,
            "div_tp": dtp, "div_fp": dfp, "div_fn": dfn, "div_jaccard": dj,
            "n_pred": len(pred_graph.nodes), "n_gt": len(gt_df), "est_nodes": est_nodes}

## Pipeline Runner

Runs detect → track → divide for a given detector function, with timing.

In [ ]:
def run_pipeline(zarr_path, detector_fn, voxel_scale=None, verbose=True):
    """Run detection + tracking on one sample. Returns TrackGraph."""
    if voxel_scale is None: voxel_scale = VOXEL_SCALE
    meta = get_array_meta(zarr_path)
    T = meta["shape"][0]
    all_centroids = {}
    for t in range(T):
        frame = load_volume_frame(zarr_path, t)
        centroids = detector_fn(frame, voxel_scale)
        all_centroids[t] = centroids
        if verbose and (t % 20 == 0 or t == T - 1):
            print(f"    t={t:3d}: {len(centroids):4d} detections")
    graph = build_tracks(all_centroids, voxel_scale)
    graph = detect_divisions(graph, all_centroids, voxel_scale)
    return graph

## Experiment: Compare All Three Detectors

For each train sample, run all three detectors and evaluate against GT.
This lets us directly compare accuracy and runtime.

In [ ]:
samples = list_samples(DATA_DIR, SPLIT)
if MAX_SAMPLES is not None:
    samples = samples[:MAX_SAMPLES]
print(f"Evaluating {len(samples)} samples: {samples}")

# Define detectors to compare
detectors = {
    "threshold": detect_cells_threshold,
}
if CELLPOSE_AVAILABLE:
    detectors["cellpose"] = detect_cells_cellpose
    detectors["hybrid"] = detect_cells_hybrid

# Results storage
all_results = []  # list of dicts

for idx, sample in enumerate(samples):
    zarr_p = os.path.join(DATA_DIR, SPLIT, f"{sample}.zarr")
    geff_p = os.path.join(DATA_DIR, SPLIT, f"{sample}.geff")
    gt = load_geff(geff_p)
    print(f"\n{'='*70}")
    print(f"[{idx+1}/{len(samples)}] {sample}")
    print(f"  GT: {len(gt['nodes_df'])} annotated nodes, {gt['edges'].shape[0]} edges")

    for det_name, det_fn in detectors.items():
        print(f"\n  --- Detector: {det_name} ---")
        t0 = time.time()
        try:
            graph = run_pipeline(zarr_p, det_fn, verbose=True)
            elapsed = time.time() - t0

            n_divs = sum(1 for n in graph.nodes if len(graph.get_outgoing_edges(n["id"])) >= 2)
            print(f"    Result: {len(graph.nodes)} nodes, {len(graph.edges)} edges, {n_divs} divs")
            print(f"    Time: {elapsed:.1f}s")

            result = evaluate_sample(graph, gt)
            print(f"    Edge Jaccard:  {result['edge_jaccard']:.4f}  (TP={result['edge_tp']} FP={result['edge_fp']} FN={result['edge_fn']})")
            print(f"    Div  Jaccard:  {result['div_jaccard']:.4f}  (TP={result['div_tp']} FP={result['div_fp']} FN={result['div_fn']})")

            result["sample"] = sample
            result["detector"] = det_name
            result["time_s"] = elapsed
            all_results.append(result)

        except Exception:
            print(f"    ERROR with {det_name}:")
            traceback.print_exc()
            all_results.append({"sample": sample, "detector": det_name,
                                "edge_jaccard": None, "div_jaccard": None, "time_s": None})

## Results Comparison

In [ ]:
results_df = pd.DataFrame(all_results)

# Per-sample breakdown
print("\n" + "="*80)
print("PER-SAMPLE RESULTS")
print("="*80)
cols = ["sample", "detector", "edge_jaccard", "div_jaccard", "n_pred", "n_gt", "time_s"]
display_cols = [c for c in cols if c in results_df.columns]
print(results_df[display_cols].to_string(index=False))

# Aggregate per detector
print("\n" + "="*80)
print("AGGREGATE BY DETECTOR")
print("="*80)
agg = results_df.groupby("detector").agg({
    "edge_jaccard": "mean",
    "div_jaccard": "mean",
    "time_s": "mean",
}).round(4)
agg.columns = ["Mean Edge Jaccard", "Mean Div Jaccard", "Mean Time (s)"]
print(agg.to_string())

# Best detector
if len(agg) > 0:
    best = agg["Mean Edge Jaccard"].idxmax()
    print(f"\n>>> Best detector by Edge Jaccard: {best} ({agg.loc[best, 'Mean Edge Jaccard']:.4f})")